### Part 1 - Data Warehouse Modeling: Star Schema

#### 1.1 Business Process and Fact Grain Definition
* **Business Process:** Monitoring and execution of financial buy/sell stock transactions (BUY/SELL).
* **Fact Table:** `Fact_Transactions`.
* **Fact Grain:** Transactional / Atomic. Each row in the fact table corresponds one-to-one with a single transaction recorded in the source file `account-statement`.
* **Design Choice:** Keeping the data at the finest atomic level of detail allows every required analytical query (e.g. aggregations by quarter, country, or sector) to be answered without any loss of information.


### 1.2 Identify Fact and Dimensions

Based on the analysis of the source datasets and following the principle of attribute non-redundancy, information has been split between the fact table (which holds the quantitative measures) and the four dimension tables (which hold the descriptive attributes).

| Table Name | Table Type | Included Attributes | Role in the Data Warehouse / Analysis |
| :--- | :--- | :--- | :--- |
| **Fact_Transactions** | Fact Table | `Quantity`, `Date_SK`, `Geography_SK` *(Nullable)*, `Symbol_SK` *(Nullable)*, `Type_SK` | Holds the numeric measure of each operation and the foreign keys to all dimensions. |
| **Dim_Time** | Dimension Table | `Date`, `Day`, `Month`, `Quarter`, `Year`, `DayOfWeek` | Enables temporal analysis and trend tracking of transactions. |
| **Dim_Geography** | Dimension Table | `Country` (or ISO Code), `Region`, `Sub-region` | Enables geopolitical segmentation of transactions based on the issuing company's headquarters. |
| **Dim_Symbol** | Dimension Table | `Symbol`, `Company`, `Sector`, `Industry` | Identifies the traded stock and its corporate classification. |
| **Dim_Transaction Type** | Dimension Table | `Transaction Type` (**BUY / SELL / DIVIDENT**) | Distinguishes the nature of the financial operation performed. |

#### Design Choices and Redundancy Elimination:

* **Separation of Master Data:** The `country` attribute originally present in `symbols.csv` serves exclusively as a logical bridge to link each stock to its official geopolitical record in `country.csv`. In the final star schema, this attribute is removed from the symbol dimension and fully absorbed into **Dim_Geography**, avoiding data duplication and ensuring consistent filtering across the dashboard.
* **Handling Nullability and Source Incompleteness:** At the logical design level, the foreign keys `Symbol_SK` and `Geography_SK` in the Fact Table have been set as **optional (nullable)**. This architectural decision was necessary because exploratory analysis revealed 18 legitimate stock tickers (e.g. European/Italian markets) not listed in the static symbols master data. Allowing null values in these relationships preserves the mathematical integrity of total volumes (`Quantity`) without polluting descriptive metrics and geopolitical filters with artificial or fictitious records.
* **Extension of Transaction Types:** The `Dim_Transaction_Type` dimension was extended to accommodate three logical states (`BUY`, `SELL`, and `DIVIDENT`) after dividend cash flows were discovered in the raw transaction log — a case not originally anticipated in the abstract theoretical model.


### 1.3 Define Dimension Hierarchies

Hierarchies establish the logical many-to-one aggregation relationships within dimensions, enabling analytical **Roll-up** (aggregating upward) and **Drill-down** (zooming into detail) operations across the system and the dashboard.

The hierarchies designed for each dimension are listed below, ordered from the most detailed level (leaf) to the most aggregated level (root):

1. **Dim_Time (Temporal Hierarchy):**
   * `Date` → `Month` → `Quarter` → `Year`
   * *Note:* The isolated attribute `DayOfWeek` is also included to support weekly seasonality analysis; it is not part of the main hierarchy chain.

2. **Dim_Geography (Geopolitical Hierarchy):**
   * `Country` (ISO code / name) → `Sub-region` → `Region`
   * *Rationale:* Allows financial flows to be analysed starting from the individual country where the issuing company is headquartered, all the way up to the macro-continental region.

3. **Dim_Symbol (Corporate Classification Hierarchy):**
   * `Symbol` → `Company` → `Industry` → `Sector`
   * *Rationale:* Supports aggregation of traded volumes by individual company, then by specific industry segment, and finally by macro-economic sector (e.g. Technology, Healthcare).

4. **Dim_Transaction Type (Flat Dimension):**
   * `Transaction Type` (**BUY / SELL / DIVIDENT**)
   * *Rationale:* No intermediate hierarchy levels exist; this dimension acts as a pure categorical selector to separate purchase flows, sale flows, and dividend income.

### 1.4 Define the Star Schema

Based on the transactional grain requirements and the hierarchies defined above, a pure **Star Schema** is implemented. To guarantee referential integrity, independence from source operational systems, and optimised aggregation query performance (JOIN), every dimension is identified by an integer **Surrogate Key (SK)**.

The central fact table holds exclusively the foreign keys to the dimensions and the quantitative measure `Quantity`, which is **fully additive** across all described hierarchies.

At the architectural level, to preserve the integrity of total financial volumes without resorting to dummy record injection (e.g. invented countries), the foreign keys `SK_Geography` and `SK_Symbol` have been designed as **optional (Nullable)**. This allows real transactions whose corporate metadata is partially incomplete at the source to be stored correctly.

#### 1. Logical Data Dictionary

* **Fact_Transactions (Fact Table)**
  * `SK_Time` (INT, Foreign Key $
ightarrow$ Dim_Time, Not Null)
  * `SK_Geography` (INT, Foreign Key $
ightarrow$ Dim_Geography, **Nullable**)
  * `SK_Symbol` (INT, Foreign Key $
ightarrow$ Dim_Symbol, **Nullable**)
  * `SK_Transaction_Type` (INT, Foreign Key $
ightarrow$ Dim_Transaction_Type, Not Null)
  * `Quantity` (INT, Fully Additive Measure — number of units exchanged in the transaction)

* **Dim_Time (Time Dimension)**
  * `SK_Time` (INT, Primary Key)
  * `Date` (DATE/DATETIME, transaction date with no intra-day granularity)
  * `Month` (INT)
  * `Quarter` (INT)
  * `Year` (INT)
  * `DayOfWeek` (VARCHAR, e.g. MONDAY, TUESDAY...)

* **Dim_Geography (Geopolitical Dimension)**
  * `SK_Geography` (INT, Primary Key)
  * `Country` (VARCHAR, official country name standardised to ISO)
  * `ISO_Code` (VARCHAR, Alpha-2 country identifier)
  * `Region` (VARCHAR, macro-continental region)
  * `Sub_Region` (VARCHAR, geopolitical sub-region)

* **Dim_Symbol (Stock Dimension)**
  * `SK_Symbol` (INT, Primary Key)
  * `Symbol` (VARCHAR, ticker / natural key, e.g. AAPL)
  * `Company` (VARCHAR, company name)
  * `Sector` (VARCHAR, macro-economic sector)
  * `Industry` (VARCHAR, specific industry segment)

* **Dim_Transaction_Type (Transaction Type Dimension)**
  * `SK_Transaction_Type` (INT, Primary Key)
  * `Transaction_Type` (VARCHAR, allowed discrete values: **BUY / SELL / DIVIDENT**)

#### 2. Star Schema Diagram (Mermaid)

The Entity-Relationship diagram of the star schema is rendered below. Note how the relationships to `Dim_Geography` and `Dim_Symbol` use zero-or-many cardinality (`}|..o|`) on the Fact Table side to graphically represent tolerance of null values for unlisted tickers.

```mermaid
erDiagram
    %% Central Fact Table
    Fact_Transactions {
        int SK_Time FK "Not Null"
        int SK_Geography FK "Nullable"
        int SK_Symbol FK "Nullable"
        int SK_Transaction_Type FK "Not Null"
        int Quantity
    }

    %% Dimension Tables
    Dim_Time {
        int SK_Time PK
        date Date
        int Month
        int Quarter
        int Year
        string DayOfWeek
    }

    Dim_Geography {
        int SK_Geography PK
        string Country
        string ISO_Code
        string Region
        string Sub_Region
    }

    Dim_Symbol {
        int SK_Symbol PK
        string Symbol
        string Company
        string Sector
        string Industry
    }

    Dim_Transaction_Type {
        int SK_Transaction_Type PK
        string Transaction_Type
    }

    %% Star Schema Relationships (Many-to-One / Zero-or-Many to One)
    Fact_Transactions }|..|| Dim_Time : "has_time"
    Fact_Transactions }|..o| Dim_Geography : "has_geography (optional)"
    Fact_Transactions }|..o| Dim_Symbol : "has_symbol (optional)"
    Fact_Transactions }|..|| Dim_Transaction_Type : "has_type"
```


In [6]:
import pandas as pd
import numpy as np

# ============================================================
# CELL 1 — EXTRACT: Load source files and structural cleanup
# ============================================================
df_transactions_raw = pd.read_csv('account-statement-1-1-2024-12-31-2024.csv', sep=';')
df_symbols_raw      = pd.read_csv('symbols.csv', sep=';')
df_country_raw      = pd.read_csv('country.csv', sep=',')

# Normalise column names across all three dataframes:
# strip whitespace, lowercase, replace spaces/hyphens with underscores,
# and drop any unnamed index columns produced by the CSV export.
for df in [df_transactions_raw, df_symbols_raw, df_country_raw]:
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')
    df.drop(columns=[col for col in df.columns if 'unnamed' in col], errors='ignore', inplace=True)

# Uppercase all string columns in the transaction log (except the raw date field)
# to guarantee consistent matching when joining against dimension lookup tables.
for col in df_transactions_raw.columns:
    if df_transactions_raw[col].dtype == 'object' and col not in ['date']:
        df_transactions_raw[col] = df_transactions_raw[col].astype(str).str.strip().str.upper()

# Convert traded units to numeric; coerce any unparseable values to NaN
# so they can be detected and handled in the quality check step.
df_transactions_raw['unit'] = pd.to_numeric(df_transactions_raw['unit'], errors='coerce')

# ── Source-level fixes applied at load time ───────────────
# Two country names in symbols.csv differ from their official ISO form in
# country.csv. Remapping them here, at the earliest possible stage, ensures
# every downstream step (quality checks, dimension build, fact table join)
# works against consistent, already-corrected data.
df_symbols_raw['country'] = df_symbols_raw['country'].replace({
    'Taiwan': 'Taiwan, Province of China',
    'Turkey': 'Türkiye',
})

# Taiwan is also missing region and sub_region data in country.csv.
# We patch it here so Dim_Geography is complete when it is built in Cell 3.
tw_mask = df_country_raw['name'].str.contains('Taiwan', case=False, na=False)
df_country_raw.loc[tw_mask, 'region']     = 'Asia'
df_country_raw.loc[tw_mask, 'sub_region'] = 'Eastern Asia'

print("=== CELL 1: EXTRACTION PIPELINE COMPLETE ===")
print(f"Raw transactions loaded: {df_transactions_raw.shape[0]} rows")
print(f"Transaction columns: {df_transactions_raw.columns.tolist()}")
print(f"Symbol columns: {df_symbols_raw.columns.tolist()}")


=== CELL 1: EXTRACTION PIPELINE COMPLETE ===
Raw transactions loaded: 2745 rows
Transaction columns: ['idtransaction', 'date', 'transactiontype', 'symbol', 'unit']
Symbol columns: ['symbol', 'company_name', 'sector', 'industry', 'country']


In [7]:
# ============================================================
# CELL 2 — TRANSFORM: Content cleaning and data quality checks
# ============================================================

# ── 2a. Remove corrupt / empty rows ───────────────────────
# Drop any row missing the three mandatory fields, then keep only
# records with a recognised transaction type.
df_trans_clean = df_transactions_raw.dropna(subset=['date', 'symbol', 'transactiontype']).copy()
df_trans_clean = df_trans_clean[df_trans_clean['date'].str.strip() != ''].copy()
df_trans_clean = df_trans_clean[df_trans_clean['transactiontype'].isin(['BUY', 'SELL', 'DIVIDENT'])].copy()

# ── 2b. Parse the date field and derive temporal attributes ─
# The raw date string includes a time component; we take only the first 10
# characters (dd/mm/yyyy) and convert them to a proper datetime object.
df_trans_clean['date_clean'] = pd.to_datetime(
    df_trans_clean['date'].str[:10], format='%d/%m/%Y', errors='coerce'
)
df_trans_clean['dayofweek'] = df_trans_clean['date_clean'].dt.day_name().str.upper()

# ── 2c. Data Quality Checks ───────────────────────────────
# Country name fixes were applied in Cell 1 at load time, so by this point
# all three checks below should find a fully consistent state.
print("=== CELL 2: DATA QUALITY CHECKS ===")

# Check 1: Missing values per column in the cleaned transaction set
print("--- Missing values per column (transactions) ---")
print(df_trans_clean.isnull().sum(), "")

# Check 2: Orphan symbols — tickers present in transactions but absent from symbols.csv.
# These are real market instruments not covered by the static master data.
sym_set        = set(df_symbols_raw['symbol'].str.strip().str.upper())
trans_syms     = set(df_trans_clean['symbol'].dropna().unique())
orphan_symbols = trans_syms - sym_set
print(f"--- Orphan symbols (not listed in symbols.csv): {len(orphan_symbols)} ---")
print(sorted(orphan_symbols), "")

# Check 3: Countries in symbols.csv that cannot be mapped to any row in country.csv.
# After the fixes applied in Cell 1 this set should be empty.
cnt_names          = set(df_country_raw['name'].str.strip())
sym_countries      = set(df_symbols_raw['country'].str.strip())
unmapped_countries = sym_countries - cnt_names
print(f"--- Countries in symbols.csv with no match in country.csv: {len(unmapped_countries)} ---")
print(sorted(unmapped_countries) if unmapped_countries else "None — mapping is complete!", "")

print(f"Clean and valid transactions: {df_trans_clean.shape[0]} rows")


=== CELL 2: DATA QUALITY CHECKS ===
--- Missing values per column (transactions) ---
idtransaction      0
date               0
transactiontype    0
symbol             0
unit               0
date_clean         0
dayofweek          0
dtype: int64 
--- Orphan symbols (not listed in symbols.csv): 18 ---
['AGO.L', 'ARCH', 'AZM', 'CCAP', 'CSIQ', 'FNC', 'HTGC', 'IBE', 'MFG', 'MONC', 'OBDC', 'RCMT', 'RIGZU', 'SAP', 'TKC', 'UCG', 'VWS', 'WF'] 
--- Countries in symbols.csv with no match in country.csv: 0 ---
None — mapping is complete! 
Clean and valid transactions: 2281 rows


In [8]:
# ============================================================
# CELL 3 — LOAD: Build the Star Schema dimension and fact tables
# ============================================================

# ── DIMENSION TABLES ─────────────────────────────────────────

# Dim_Transaction_Type: a small, flat lookup for the three allowed operation types.
df_dim_type = pd.DataFrame({'transaction_type': ['BUY', 'SELL', 'DIVIDENT']})
df_dim_type.insert(0, 'sk_transaction_type', df_dim_type.index + 1)

# Dim_Geography: derived from country.csv, which already has clean ISO data.
# Taiwan's region and sub_region were patched in Cell 2.
df_dim_geo = (df_country_raw[['name', 'alpha_2', 'region', 'sub_region']]
              .drop_duplicates().reset_index(drop=True)
              .rename(columns={'name': 'country', 'alpha_2': 'iso_code'}))
df_dim_geo.insert(0, 'sk_geography', df_dim_geo.index + 1)

# Dim_Symbol: stock master data containing ticker, company name, sector, and industry.
# The country attribute is intentionally excluded — it belongs to Dim_Geography only.
df_dim_symbol = (df_symbols_raw[['symbol', 'company_name', 'sector', 'industry']]
                 .drop_duplicates().reset_index(drop=True))
df_dim_symbol.insert(0, 'sk_symbol', df_dim_symbol.index + 1)

# Dim_Time: built from the actual transaction dates rather than a pre-generated
# calendar, so it contains only the 248 trading days present in the dataset.
unique_dates = pd.DataFrame({'date': df_trans_clean['date_clean'].dropna().unique()})
df_dim_time  = unique_dates.sort_values('date').reset_index(drop=True).copy()
df_dim_time.insert(0, 'sk_time', df_dim_time.index + 1)
df_dim_time['day']       = df_dim_time['date'].dt.day
df_dim_time['month']     = df_dim_time['date'].dt.month
df_dim_time['quarter']   = df_dim_time['date'].dt.quarter
df_dim_time['year']      = df_dim_time['date'].dt.year
df_dim_time['dayofweek'] = df_dim_time['date'].dt.day_name().str.upper()

# ── FACT TABLE ───────────────────────────────────────────────
# Build SK lookup tables for each dimension, then join them onto the
# cleaned transaction set to produce the final star schema fact table.
time_lookup    = df_dim_time[['sk_time', 'date']].drop_duplicates(subset=['date'])
symbol_lookup  = df_dim_symbol[['sk_symbol', 'symbol']].drop_duplicates(subset=['symbol'])
type_lookup    = df_dim_type[['sk_transaction_type', 'transaction_type']].drop_duplicates()
geo_lookup     = df_dim_geo[['sk_geography', 'country']].drop_duplicates(subset=['country'])
sym_to_country = df_symbols_raw[['symbol', 'country']].drop_duplicates(subset=['symbol'])

df_fact = df_trans_clean.copy()
df_fact = pd.merge(df_fact, sym_to_country,  on='symbol',          how='left')
df_fact = pd.merge(df_fact, time_lookup,     left_on='date_clean', right_on='date', how='left')
df_fact = pd.merge(df_fact, symbol_lookup,   on='symbol',          how='left')
df_fact = pd.merge(df_fact, geo_lookup,      on='country',         how='left')
df_fact = pd.merge(df_fact, type_lookup,     left_on='transactiontype', right_on='transaction_type', how='left')

df_fact.rename(columns={'unit': 'quantity'}, inplace=True)

# idtransaction is kept as a degenerate dimension — it has no dedicated
# dimension table but provides a direct audit trail back to the source system.
fact_transactions = df_fact[['idtransaction', 'sk_time', 'sk_geography', 'sk_symbol',
                              'sk_transaction_type', 'quantity']].copy()

# Cast all surrogate keys to nullable integer so that orphan records
# (sk_geography / sk_symbol = NaN) are stored correctly without type errors.
for col in ['sk_time', 'sk_geography', 'sk_symbol', 'sk_transaction_type']:
    fact_transactions[col] = fact_transactions[col].astype('Int64')

print("=== CELL 3: STAR SCHEMA BUILT ===")
print(f"Rows in Fact_Transactions: {fact_transactions.shape[0]}")
print("Missing values (nullable FKs expected for orphan tickers):")
print(fact_transactions.isnull().sum())


=== CELL 3: STAR SCHEMA BUILT ===
Rows in Fact_Transactions: 2281
Missing values (nullable FKs expected for orphan tickers):
idtransaction            0
sk_time                  0
sk_geography           212
sk_symbol              212
sk_transaction_type      0
quantity                 0
dtype: int64


In [ ]:
# ============================================================
# CELL 4 — ANALYTICAL QUERIES (Part 2.2)
# Answer 5 questions from the official homework list using SQL
# executed against an in-memory SQLite database built on top of
# the star schema dataframes created in Cell 3.
# ============================================================
import sqlite3

# Load all star schema tables into an in-memory SQLite database.
# This approach lets us write standard SQL JOINs directly on the
# pandas dataframes without any external database dependency.
conn = sqlite3.connect(':memory:')
fact_transactions.to_sql('Fact_Transactions',    conn, index=False, if_exists='replace')
df_dim_symbol.to_sql('Dim_Symbol',              conn, index=False, if_exists='replace')
df_dim_geo.to_sql('Dim_Geography',              conn, index=False, if_exists='replace')
df_dim_type.to_sql('Dim_Transaction_Type',      conn, index=False, if_exists='replace')
df_dim_time.to_sql('Dim_Time',                  conn, index=False, if_exists='replace')

print("In-memory SQL database ready. Running 5 analytical queries.")
print("=" * 60)

# ── QUESTION 1 ────────────────────────────────────────────
# Top 5 sectors by number of SELL transactions in the United States during 2024.
print("QUESTION 1 — Top 5 sectors by SELL transactions in United States")
q1 = """
SELECT s.sector, COUNT(*) AS n_transactions
FROM Fact_Transactions f
JOIN Dim_Symbol s ON f.sk_symbol = s.sk_symbol
JOIN Dim_Geography g ON f.sk_geography = g.sk_geography
JOIN Dim_Transaction_Type t ON f.sk_transaction_type = t.sk_transaction_type
WHERE t.transaction_type = 'SELL'
  AND g.country = 'United States of America'
GROUP BY s.sector
ORDER BY n_transactions DESC
LIMIT 5;
"""
print(pd.read_sql_query(q1, conn).to_string(index=False))
print()

# ── QUESTION 2 ────────────────────────────────────────────
# Top 5 industries by number of BUY transactions in Q4 2024.
print("QUESTION 2 — Top 5 industries by BUY transactions in Q4 2024")
q2 = """
SELECT s.industry, COUNT(*) AS n_transactions
FROM Fact_Transactions f
JOIN Dim_Symbol s ON f.sk_symbol = s.sk_symbol
JOIN Dim_Transaction_Type t ON f.sk_transaction_type = t.sk_transaction_type
JOIN Dim_Time dt ON f.sk_time = dt.sk_time
WHERE t.transaction_type = 'BUY'
  AND dt.quarter = 4
GROUP BY s.industry
ORDER BY n_transactions DESC
LIMIT 5;
"""
print(pd.read_sql_query(q2, conn).to_string(index=False))
print()

# ── QUESTION 3 ────────────────────────────────────────────
# Rank all quarters of 2024 by total number of transactions (BUY + SELL).
# Window function RANK() assigns tied quarters the same position.
print("QUESTION 3 — Quarters ranked by total transactions (BUY + SELL)")
q3 = """
SELECT dt.quarter AS quarter, COUNT(*) AS n_transactions
FROM Fact_Transactions AS f
JOIN Dim_Transaction_Type AS t ON f.sk_transaction_type = t.sk_transaction_type
JOIN Dim_Time AS dt ON f.sk_time = dt.sk_time
WHERE t.transaction_type IN ('BUY', 'SELL')
GROUP BY dt.quarter
ORDER BY n_transactions DESC;
"""
print(pd.read_sql_query(q3, conn).to_string(index=False))
print()

# ── QUESTION 5 ────────────────────────────────────────────
# Top 5 regions by total units bought across 2024.
print("QUESTION 5 — Top 5 regions by total units bought")
q5 = """
SELECT g.region, SUM(f.quantity) AS total_units_bought
FROM Fact_Transactions f
JOIN Dim_Geography g ON f.sk_geography = g.sk_geography
JOIN Dim_Transaction_Type t ON f.sk_transaction_type = t.sk_transaction_type
WHERE t.transaction_type = 'BUY'
GROUP BY g.region
ORDER BY total_units_bought DESC
LIMIT 5;
"""
print(pd.read_sql_query(q5, conn).to_string(index=False))
print()

# ── QUESTION 7 ────────────────────────────────────────────
# Top 10 symbols by total number of transactions (BUY + SELL) in 2024.
print("QUESTION 7 — Top 10 symbols by number of transactions (BUY + SELL)")
q7 = """
SELECT s.symbol, s.company_name, COUNT(*) AS n_transactions
FROM Fact_Transactions f
JOIN Dim_Symbol s ON f.sk_symbol = s.sk_symbol
JOIN Dim_Transaction_Type t ON f.sk_transaction_type = t.sk_transaction_type
WHERE t.transaction_type IN ('BUY', 'SELL')
GROUP BY s.symbol, s.company_name
ORDER BY n_transactions DESC
LIMIT 10;
"""
print(pd.read_sql_query(q7, conn).to_string(index=False))
conn.close()


In-memory SQL database ready. Running 5 analytical queries.
QUESTION 1 — Top 5 sectors by SELL transactions in United States
                sector  n_transactions
            Technology             158
Communication Services              58
    Financial Services              55
            Healthcare              50
     Consumer Cyclical              48

QUESTION 2 — Top 5 industries by BUY transactions in Q4 2024
                      industry  n_transactions
                Semiconductors              18
Internet Content & Information              15
     Software - Infrastructure              10
               Internet Retail               8
        Diagnostics & Research               7

QUESTION 3 — Quarters ranked by total transactions (BUY + SELL)
 quarter  n_transactions
       1            1076
       2             580
       3             260
       4             255

QUESTION 5 — Top 5 regions by total units bought
  region  total_units_bought
Americas             37026.0